# Task 4 – Teil 3: Spannungsband, Auslegung des Ortsnetzes und Schalterstellungen

**Ausgangspunkt:** CIGRE-MV-Netz ohne DER mit dem CIGRE-LV-Ortsnetz an Bus 12 (Ergebnis von Teil 2).

**Vorgehen:**

1. **Spannungsband festlegen:** Arbeitsgrenzen 0,90 bis 1,10 pu und ein engeres planerisches Spannungsband mit Marge.
2. **Leistungsspitzen der Ortsnetztrafos ermitteln:** Die Scheinleistung jedes Ortsnetztrafos wird Stunde für Stunde dargestellt. Daraus wird der passende pandapower-Standardtrafo gewählt.
3. **Stromspitzen der überlasteten Leitungen ermitteln:** Line C1-C2 und Line C2-C3 im Gewerbenetz sind in Teil 2 über 100 % ausgelastet. Eine Auslastung über 100 % ist im Dauerbetrieb nicht zulässig. Aus der Stromspitze wird eine passende pandapower-Standardleitung gewählt.
4. **Trafos und Leitungen ersetzen:** Bei der Auswahl bleibt genug Reserve für Leistungs- und Stromspitzen, denn die Simulation bildet nur den Normalbetrieb ab (Standardlastprofile ohne Spitzen einzelner Kunden, Stundenwerte, keine unvorhersehbaren Situationen).
5. **Schalterstellungen:** Das ausgelegte Netz wird wie in Teil 1 für alle 8 Stellungen der Trennschalter S1–S3 untersucht, statisch und als Zeitreihe.

Die automatische Regelung des Ortsnetztransformators folgt im nächsten Schritt.

## Importe

In [ ]:
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn

import simulation as sim
import szenarien as sz
import lvnetz
import auslegung as al
import netplan
import plots
from elements import Diagram, Profile
from profiles import Month

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.tab20.colors)

# Zeitreihen nur neu rechnen, wenn sich Netz, Einstellungen oder Daten geändert haben
sim.ONLY_RUN_IF_CHANGED = True

def baseline_net():
    """Unverändertes CIGRE-MV-Netz ohne dezentrale Erzeuger (Referenz aus Teil 1)."""
    return pn.create_cigre_network_mv(with_der=False)

ANSCHLUSSBUS = "Bus 12"                              # Ergebnis von Teil 2

Neues Modul `auslegung.py`: Spitzenwerte je Element (`spitzen`) und passende pandapower-Standardtrafos und -leitungen (`trafo_kandidaten`, `leitung_kandidaten`). Neue Diagrammarten: `Diagram.ACTIVE_POWER` (Wirkleistung), `Diagram.APPARENT_POWER` (Scheinleistung) und `Diagram.CURRENT` (Strom) für beliebige Elemente (Trafos, Leitungen, Busse, Lasten, Erzeuger, externes Netz), mit gestrichelter Linie beim Bemessungswert.

## 1 Spannungsband

**Arbeitsgrenzen 0,90 bis 1,10 pu:** Vorgabe des Arbeitsauftrags. In Verteilnetzen dürfen die Knotenspannungen betriebsbedingt bis zu 10 % nach oben oder unten abweichen [19, S. 712]; DIN EN 50160 legt für Nieder- und Mittelspannung ±10 % fest [15]. Eine Verletzung bedeutet unzulässigen Betrieb.

**Planerisches Spannungsband 0,95 bis 1,05 pu:** Zielband für die Auslegung und die spätere Regelung. Die Marge von 5 % zu den Arbeitsgrenzen deckt ab, was die Simulation nicht abbildet:

- Die BDEW-Profile beschreiben den Durchschnitt vieler Kunden; Spitzen einzelner Kunden (Durchlauferhitzer, Wallbox, Wärmepumpe) fehlen.
- Die Simulation rechnet mit Stundenwerten, DIN EN 50160 bewertet 10-Minuten-Mittelwerte; kurze Spitzen werden geglättet.
- Der Lastfluss ist symmetrisch; im Niederspannungsnetz sind die Phasen in der Realität ungleich belastet.
- Je Monat wird nur eine typische Woche betrachtet; Extremwetter und künftige Lasten (E-Mobilität, Wärmepumpen, weitere PV) fehlen.
- Ein Stufenschalter stellt die Spannung nur in Stufen von 2,5 % und mit Verzögerung.

Damit folgt das Planungsband dem Grundsatz, Sollwerte so zu setzen, dass die Werte „möglichst wieder mittig in den Toleranzbändern liegen“ [19, S. 716]. Die Bandbreite von ±5 % ist zudem deutlich größer als eine Stufe des Stufenschalters (2,5 %); ein Regler kann die Spannung damit ins Band bringen, ohne ständig hin- und herzuschalten.

| Band | untere Grenze | obere Grenze | Bedeutung |
|---|---|---|---|
| Arbeitsgrenzen | 0,90 pu | 1,10 pu | unzulässiger Betrieb bei Verletzung |
| Planungsband | 0,95 pu | 1,05 pu | Ziel für Auslegung und Regelung; Verletzung = Reserve aufgebraucht |

In [ ]:
V_MIN, V_MAX = 0.90, 1.10                 # Arbeitsgrenzen
V_PLAN_MIN, V_PLAN_MAX = 0.95, 1.05       # planerisches Spannungsband

# Markierungen für Spannungsdiagramme (Arbeitsgrenzen zeichnet die Software selbst rot)
PLAN_MARKER = [("orange", V_PLAN_MIN, "Planungsband 0,95 pu", "--"), ("orange", V_PLAN_MAX, "Planungsband 1,05 pu", "--")]

def band_stunden(s):
    """Stunden außerhalb des Planungsbands und der Arbeitsgrenzen, getrennt nach MS und NS."""
    g = lvnetz.bus_groups(s.net)
    vm = s.results.vm_pu[s.results.converged]
    out = {}
    for lvl, cols in (("MS", g["MS"]), ("NS", g["NS"])):
        if not cols:
            continue
        lo, hi = vm[cols].min(axis=1), vm[cols].max(axis=1)
        out[f"Stunden < {V_PLAN_MIN} {lvl}"] = int((lo < V_PLAN_MIN).sum())
        out[f"Stunden > {V_PLAN_MAX} {lvl}"] = int((hi > V_PLAN_MAX).sum())
        out[f"Stunden < {V_MIN} {lvl}"] = int((lo < V_MIN).sum())
        out[f"Stunden > {V_MAX} {lvl}"] = int((hi > V_MAX).sum())
    return pd.Series(out)

## 2 Ausgangslage: Ortsnetz an Bus 12

In [ ]:
ref = sim.init("w4_baseline", baseline_net(), overwrite=True, assignProfiles=True)
vorher = ref.copy("w4_lv_Bus12", overwrite=True)
lvnetz.attach(vorher, ANSCHLUSSBUS)
vorher.runSimulation()
w, o = lvnetz.kennwerte_sim(vorher)
display(pd.DataFrame({"Wert": w, "Ort": o}).dropna(subset=["Wert"]))
band_stunden(vorher).to_frame("Ortsnetz an Bus 12 (CIGRE-Betriebsmittel)")

In [ ]:
ns_busse = lvnetz.bus_groups(vorher.net)["NS"]
schwach = vorher.results.vm_pu[ns_busse].min().nsmallest(3).index.tolist()
vorher.createDiagram(Diagram.VOLTAGE, schwach, months=Month.JANUARY, markers=PLAN_MARKER,
                     title="Niedrigste NS-Spannungen im Januar (CIGRE-Betriebsmittel)")

## 3 Leistungsspitzen der Ortsnetztrafos

Die Trafos werden nach der **Scheinleistung** ausgelegt, weil Wicklungen und Kühlung durch den Strom belastet werden und dieser von der Scheinleistung abhängt. Die gestrichelten Linien zeigen die Bemessungsleistung $S_r$ der aktuell verbauten CIGRE-Trafos.

In [ ]:
TRAFOS = ["LV Trafo R0-R1", "LV Trafo I0-I1", "LV Trafo C0-C1"]
vorher.createDiagram(Diagram.APPARENT_POWER, TRAFOS, layout="overview", title="Scheinleistung der Ortsnetztrafos")
vorher.createDiagram(Diagram.APPARENT_POWER, TRAFOS, months=Month.JANUARY, title="Scheinleistung der Ortsnetztrafos")
vorher.createDiagram(Diagram.ACTIVE_POWER, TRAFOS, months=Month.JANUARY, title="Wirkleistung der Ortsnetztrafos (OS-Seite)")

In [ ]:
spitzen_trafo = al.spitzen(vorher, TRAFOS)
spitzen_trafo

**Auswahlregel:** Gewählt wird der kleinste pandapower-Standardtrafo 20/0,4 kV, bei dem die Spitzenlast höchstens **70 %** der Bemessungsleistung erreicht (`MAX_AUSLASTUNG`). Die Reserve von 30 % deckt Leistungsspitzen, die im Normalbetrieb mit Standardlastprofilen nicht enthalten sind, und künftigen Lastzuwachs ab. Verfügbare Standardtypen: 0,25 MVA, 0,4 MVA und 0,63 MVA.

In [ ]:
MAX_AUSLASTUNG = 0.70

trafo_wahl = {}
for t in TRAFOS:
    k = al.trafo_kandidaten(vorher.net, spitzen_trafo.at[t, "S max [MVA]"], MAX_AUSLASTUNG)
    trafo_wahl[t] = al.kleinster_geeigneter(k)
    print(f"{t}: S max = {spitzen_trafo.at[t, 'S max [MVA]']:.3f} MVA → {trafo_wahl[t]}")
    display(k)

## 4 Stromspitzen der Leitungen

Strom auf Line C1-C2 und Line C2-C3 (Freileitung OH1, Bemessungsstrom 172 A nach [18]) über alle 12 Wochen und in der Januarwoche:

In [ ]:
LEITUNGEN = ["LV Line C1-C2", "LV Line C2-C3"]
vorher.createDiagram(Diagram.CURRENT, LEITUNGEN, layout="overview", title="Strom auf Line C1-C2 und C2-C3")
vorher.createDiagram(Diagram.CURRENT, LEITUNGEN, months=Month.JANUARY, title="Strom auf Line C1-C2 und C2-C3")
vorher.createDiagram(Diagram.CURRENT, LEITUNGEN, layout="boxplot", title="Strom auf Line C1-C2 und C2-C3")

Zur Einordnung die Spitzenwerte **aller** LV-Leitungen, sortiert nach der höchsten Auslastung:

In [ ]:
lv_leitungen = lvnetz.element_groups(vorher.net)["line_NS"]
spitzen_ltg = al.spitzen(vorher, lv_leitungen).sort_values("Auslastung max [%]", ascending=False)
plots.table(spitzen_ltg.drop(columns=["S_r [MVA]"]), decimals=1)

**Auswahlregel:** Die Leitungen sind Freileitungen; ersetzt wird deshalb durch eine pandapower-Standard-Freileitung der Niederspannung (`voltage_rating = "LV"`, `type = "ol"`). Gewählt wird die kleinste, bei der der Spitzenstrom höchstens 70 % des Bemessungsstroms erreicht.

In [ ]:
leitung_wahl = {}
for l in LEITUNGEN:
    k = al.leitung_kandidaten(vorher.net, spitzen_ltg.at[l, "I max [A]"], art="ol", max_auslastung=MAX_AUSLASTUNG)
    leitung_wahl[l] = al.kleinster_geeigneter(k)
    print(f"{l}: I max = {spitzen_ltg.at[l, 'I max [A]']:.0f} A → {leitung_wahl[l]}")
display(k)

## 5 Trafos und Leitungen ersetzen

Das ausgelegte Netz ist eine Kopie der Ausgangslage, in der nur die Typen der drei Ortsnetztrafos und der beiden Leitungen geändert werden (`changeParameter(..., "std_type", ...)` übernimmt alle Typparameter). Alles andere bleibt gleich.

**Folgen der Standardtrafos:** $u_k$ = 6 % statt 4,1 %, Eisen- und Leerlaufverluste, Schaltgruppe mit 150° Phasenverschiebung (die NS-Winkel verschieben sich deshalb gegenüber Teil 2) und ein Stufenschalter auf der OS-Seite mit ±2 × 2,5 %, Stellung 0. Bei Verteiltransformatoren ist dieser Stellbereich üblicherweise ein Umsteller, der nur stromlos verstellt wird [19, S. 424]; für die Regelung im nächsten Schritt wird derselbe Stellbereich als Laststufenschalter (regelbarer Ortsnetztrafo) genutzt.

In [ ]:
ausgelegt = vorher.copy("w4_t3_ausgelegt", overwrite=True)
for name, typ in {**trafo_wahl, **leitung_wahl}.items():
    ausgelegt.changeParameter(name, "std_type", typ)
display(ausgelegt.changedParameters())
ausgelegt.runSimulation()

**Vergleich vorher/nachher** (gleiche Randbedingungen, nur die Betriebsmitteltypen geändert):

In [ ]:
vergl = pd.DataFrame({
    "vorher (CIGRE)": pd.concat([lvnetz.kennwerte_sim(vorher)[0], band_stunden(vorher)]),
    "nachher (Standardtypen)": pd.concat([lvnetz.kennwerte_sim(ausgelegt)[0], band_stunden(ausgelegt)]),
})
vergl.index.name = "Kennwert"
plots.table(vergl.astype(float), decimals=3)

In [ ]:
display(pd.concat({"vorher": al.spitzen(vorher, TRAFOS + LEITUNGEN)[["S max [MVA]", "I max [A]", "Auslastung max [%]", "Typ"]],
                   "nachher": al.spitzen(ausgelegt, TRAFOS + LEITUNGEN)[["S max [MVA]", "I max [A]", "Auslastung max [%]", "Typ"]]}, axis=1))
ausgelegt.createDiagram(Diagram.APPARENT_POWER, TRAFOS, layout="overview", title="Scheinleistung der Ortsnetztrafos (Standardtypen)")
ausgelegt.createDiagram(Diagram.CURRENT, LEITUNGEN, months=Month.JANUARY, title="Strom auf Line C1-C2 und C2-C3 (Standardtyp)")

In [ ]:
schwach = ausgelegt.results.vm_pu[ns_busse].min().nsmallest(3).index.tolist()
ausgelegt.createDiagram(Diagram.VOLTAGE, schwach, months=Month.JANUARY, markers=PLAN_MARKER,
                        title="Niedrigste NS-Spannungen im Januar (Standardtypen)")
ausgelegt.createDiagram(Diagram.NETWORK, time=["min", "max_loading"], title="Ausgelegtes Netz")

**Ergebnis der Auslegung (Zeitreihe, alle Trennschalter offen):**

| Betriebsmittel | vorher (CIGRE) | Spitze | nachher (pandapower-Standardtyp) | höchste Auslastung nachher |
|---|---|---|---|---|
| LV Trafo R0-R1 | 0,50 MVA, 84,0 % | 0,423 MVA (So 12.01. 18:00) | 0,63 MVA 20/0.4 kV | 66,8 % |
| LV Trafo I0-I1 | 0,15 MVA, 69,9 % | 0,106 MVA (Fr 10.01. 10:00) | 0,25 MVA 20/0.4 kV | 41,8 % |
| LV Trafo C0-C1 | 0,30 MVA, 84,3 % | 0,255 MVA (Fr 10.01. 10:00) | 0,4 MVA 20/0.4 kV | 63,1 % |
| LV Line C1-C2, C2-C3 | OH1, 172 A, 110,3 % | 190 A (Fr 10.01. 10:00) | 94-AL1/15-ST1A 0.4, 350 A | 53,8 % |

- Das Wohngebiet hat seine Spitze am Winterabend (Haushaltsprofil H25), Industrie und Gewerbe am Werktagvormittag (G25).
- Beim Wohngebiet reicht der 0,4-MVA-Trafo nicht (Spitze 106 %), beim Gewerbe nicht der 0,25-MVA-Trafo (102 %). Für die Industrie ist schon der kleinste Standardtyp mit 42 % reichlich bemessen.
- Bei der Leitung wäre 48-AL1/8-ST1A (210 A) mit 90 % zu knapp; gewählt wird 94-AL1/15-ST1A mit 54 %.
- Nach dem Tausch ist kein Betriebsmittel mehr über 100 % ausgelastet. Die höchste Auslastung im Ortsnetz haben jetzt die Kabel LV Line R1-R2 und R2-R3 (UG1, 430 A) mit 73 %; sie liegen knapp über der Auslegungsreserve von 70 %, aber deutlich unter 100 %, und wurden deshalb nicht getauscht.
- **Spannung:** Die niedrigste NS-Spannung bleibt bei 0,923 pu (LV Bus C12), die Stunden unter 0,95 pu sinken von 535 auf 450. Der größere Leiterquerschnitt im Gewerbenetz verringert den Spannungsfall, der höhere $u_k$ der Standardtrafos (6 % statt 4,1 %) wird durch ihre geringere Auslastung weitgehend ausgeglichen. Die Arbeitsgrenzen werden nicht verletzt, das Planungsband aber in 450 Stunden unterschritten; das ist die Aufgabe für die Regelung.
- **Winkel:** Durch die Schaltgruppe Dyn5/Yzn5 (150°) liegen die NS-Winkel jetzt bei etwa +173° bis +179° statt −61° bis −67° (−30° − 150° = −180°, als Winkel zwischen −180° und +180° dargestellt). Das ist eine reine Bezugsverschiebung und keine physikalische Änderung des Leistungsflusses.

## 6 Alle Schalterstellungen S1–S3

Aufbau wie in Teil 1: Jeder Schaltzustand startet von einer Kopie des ausgelegten Netzes; geändert wird nur die Stellung der Trennschalter S1–S3 im MV-Netz (Schlüssel S1 S2 S3, 1 = geschlossen). Die Schalter LV S1–S3 im Ortsnetz bleiben geschlossen.

### 6.1 Statischer Lastfluss

In [ ]:
static = sz.statisch_alle(lambda: copy.deepcopy(ausgelegt.net))
sz.konvergenz(static)

In [ ]:
bil = sz.bilanzen(static)
display(plots.table(bil.drop(columns=["Bus U min", "Leitung mit max"]).astype(float).T, decimals=3))
bil[["Bus U min", "Leitung mit max"]].T

In [ ]:
g = lvnetz.bus_groups(ausgelegt.net)
e = lvnetz.element_groups(ausgelegt.net)
vm_all = sz.vergleich(static, "vm_pu")
sz.heatmap(vm_all.loc[g["MS"]], "Spannung MS-Busse je Schaltzustand (statisch)", "Spannung [pu]", cmap="RdYlGn", vmin=0.90, vmax=1.03)
plt.show()
sz.heatmap(vm_all.loc[g["NS"]], "Spannung NS-Busse je Schaltzustand (statisch)", "Spannung [pu]", cmap="RdYlGn", vmin=0.85, vmax=1.03)
plt.show()

In [ ]:
va_all = sz.vergleich(static, "va_degree")
sz.heatmap(va_all.loc[g["MS"]], "Spannungswinkel MS-Busse (statisch, bezogen auf Bus 0)", "Winkel [°]", fmt="{:.2f}")
plt.show()
sz.heatmap(va_all.loc[g["NS"]], "Spannungswinkel NS-Busse (statisch, bezogen auf Bus 0)", "Winkel [°]", fmt="{:.2f}")
plt.show()

In [ ]:
ll_all = sz.vergleich(static, "loading_percent")
sz.heatmap(ll_all.loc[e["line_MS"]], "Auslastung MS-Leitungen (statisch)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()
sz.heatmap(ll_all.loc[e["line_NS"]], "Auslastung NS-Leitungen (statisch)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()
sz.heatmap(sz.vergleich(static, "trafo_loading"), "Trafoauslastung (statisch)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()
p_all = sz.vergleich(static, "p_from_mw")
lim = np.nanmax(np.abs(p_all.loc[e["line_MS"]].to_numpy()))
sz.heatmap(p_all.loc[e["line_MS"]], "Wirkleistung am Leitungsanfang, MS (> 0: from_bus → to_bus)", "Wirkleistung [MW]",
           cmap="RdBu_r", vmin=-lim, vmax=lim, fmt="{:.2f}")
plt.show()

In [ ]:
for key, r in static.items():
    netplan.plot_static(r["net"], title=f"Ausgelegtes Netz, statischer Lastfluss – {sz.label(key)}", elements=False)

### 6.2 Zeitreihe

In [ ]:
ts = sz.zeitreihe_alle(ausgelegt, prefix="w4_t3")

In [ ]:
kz = sz.kennzahlen(ts)
text_cols = ["konvergiert", "Bus U min", "Zeitpunkt U min", "Leitung", "Zeitpunkt Leitung max"]
display(kz[text_cols])
display(plots.table(kz.drop(columns=text_cols).astype(float).T, decimals=3))
pd.DataFrame({sz.label(k, kurz=True): band_stunden(s) for k, s in ts.items()})

In [ ]:
vmin_ts = sz.je_element(ts, "vm_pu", "min")
sz.heatmap(vmin_ts.loc[g["MS"]], "Niedrigste Spannung MS-Busse (Zeitreihe)", "Spannung [pu]", cmap="RdYlGn", vmin=0.90, vmax=1.03)
plt.show()
sz.heatmap(vmin_ts.loc[g["NS"]], "Niedrigste Spannung NS-Busse (Zeitreihe)", "Spannung [pu]", cmap="RdYlGn", vmin=0.85, vmax=1.03)
plt.show()
llmax_ts = sz.je_element(ts, "line_loading", "max")
sz.heatmap(llmax_ts.loc[e["line_MS"]], "Höchste Auslastung MS-Leitungen (Zeitreihe)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()
sz.heatmap(llmax_ts.loc[e["line_NS"]], "Höchste Auslastung NS-Leitungen (Zeitreihe)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()
sz.heatmap(sz.je_element(ts, "trafo_loading", "max"), "Höchste Trafoauslastung (Zeitreihe)", "Auslastung [%]", cmap="YlOrRd", vmin=0, vmax=100, fmt="{:.1f}", grenze=100)
plt.show()

In [ ]:
schwach = vmin_ts.loc[g["NS"]].min(axis=1).idxmin()
sim.compare([ts[k] for k in sz.KEYS], Diagram.VOLTAGE, [schwach], months=Month.JANUARY, markers=PLAN_MARKER,
            title=f"Spannung an {schwach} – alle Schaltzustände")
sim.compare([ts["000"], ts["100"]], Diagram.TRAFO_LOADING, None, months=Month.JANUARY,
            title="Trafoauslastung – alle offen (000) und S1 zu (100)")

In [ ]:
ts["000"].createDiagram(Diagram.NETWORK, time="max_loading", title="Ausgelegtes Netz – alle Trennschalter offen")
ts["100"].createDiagram(Diagram.NETWORK, time="max_loading", title="Ausgelegtes Netz – S1 geschlossen")

**Ergebnis Schalterstellungen (ausgelegtes Netz, Ortsnetz an Bus 12):**

- **Konvergenz:** Alle 8 Zustände konvergieren statisch und in allen 8 × 2016 Zeitpunkten.
- **Statisch (alle Lasten mit Bemessungsleistung):** Wie in Teil 1 ist Trafo 0-1 ohne S1 mit 101,4 % leicht überlastet und Line 2-3 mit 97 % hoch belastet; mit S1 sinken beide auf 96,6 % bzw. 74 %. Die niedrigste Spannung liegt im Ortsnetz (LV Bus R15) bei 0,913 pu, mit S1 bei 0,910 pu.
- **Zeitreihe:** In keinem Zustand werden die Arbeitsgrenzen verletzt oder ein Betriebsmittel über 100 % belastet. Die Ortsnetztrafos bleiben bei höchstens 67 %, die LV-Leitungen bei 73 %.
- **Wirkung von S1:** S1 entlastet Abgang 1 (Line 1-2/2-3: 78,6 % → 42 %, Trafo 0-1: 82,7 % → 75 %) und senkt die Verluste um rund 21 % (163 → 129 MWh). Gleichzeitig übernimmt Trafo 0-12 einen Teil der Last von Abgang 1 (70,7 % → 77,8 %). Weil das Ortsnetz an Bus 12 hängt, wird es dadurch geringfügig schlechter versorgt: Die niedrigste NS-Spannung sinkt von 0,923 auf 0,921 pu, die Stunden unter 0,95 pu steigen von 450 auf 473.
- **S2 und S3** verschieben wie in Teil 1 nur die Aufteilung innerhalb von Abgang 1 und haben auf das Ortsnetz keinen Einfluss.
- **Folgerung:** Für das Ortsnetz ist der Normalbetrieb mit offenen Trennschaltern sogar etwas günstiger als der gekuppelte Betrieb. Die Unterschreitungen des Planungsbands im Ortsnetz (450 bis 473 Stunden) lassen sich durch Schalthandlungen im MV-Netz nicht beheben; sie entstehen im Ortsnetz selbst und sind die Aufgabe des regelbaren Ortsnetztransformators.

### Quellen

[13] K. Strunz et al., „Benchmark Systems for Network Integration of Renewable and Distributed Energy Resources“, CIGRE Task Force C6.04.02, Technical Brochure 575, Paris: CIGRE, 2014.

[14] L. Thurner et al., „pandapower – An Open-Source Python Tool for Convenient Modeling, Analysis, and Optimization of Electric Power Systems“, IEEE Transactions on Power Systems, Bd. 33, Nr. 6, S. 6510–6521, Nov. 2018, doi: 10.1109/TPWRS.2018.2829021.

[15] DIN EN 50160, „Merkmale der Spannung in öffentlichen Elektrizitätsversorgungsnetzen“ (EN 50160:2010 mit Änderungen), Berlin: Beuth.

[18] *Smart Energy Management for Smart Grid*, Basel: MDPI Books, Tabelle der Leitungstypen des CIGRE-LV-Netzes. Online: https://mdpi-res.com/bookfiles/book/7980/Smart_Energy_Management_for_Smart_Grid.pdf

[19] A. J. Schwab, *Elektroenergiesysteme: Erzeugung, Transport, Übertragung und Verteilung elektrischer Energie*, 3. Aufl. Berlin, Heidelberg: Springer, 2012, doi: 10.1007/978-3-642-21958-0.